𝐐𝐮𝐞𝐬𝐭𝐢𝐨𝐧:

You have a dataset of transactions that contains the following fields:

* transaction_id (integer): Unique ID for each transaction.
* user_id (integer): ID of the user performing the transaction.
* transaction_amount (float): Amount of the transaction.
* transaction_date (string): Date of the transaction in yyyy-MM-dd format.

From this dataset, perform the following operations:

->Find the top 3 users with the highest total transaction amounts.

->Among these top 3 users, for each, identify the most recent transaction date.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

In [0]:
data = [ (1, 101, 500.0, "2024-01-01"), (2, 102, 200.0, "2024-01-02"), 
(3, 101, 300.0, "2024-01-03"), (4, 103, 100.0, "2024-01-04"), 
(5, 102, 400.0, "2024-01-05"), (6, 103, 600.0, "2024-01-06"), 
(7, 101, 200.0, "2024-01-07"), ] 

columns = ["transaction_id", "user_id", "transaction_amount", "transaction_date"]

df=spark.createDataFrame(data,columns)
display(df)

In [0]:
#Total transaction amount per user + recent date
user_summery = df.groupBy("user_id").agg(F.max("transaction_date").alias("recent_date"),F.sum("transaction_amount").alias("total_amount"))
display(user_summery)

In [0]:
#Rank user by total amount
windowSpec = Window.partitionBy("total_amount").orderBy(F.desc("total_amount"))
ranked_user = user_summery.withColumn("rank", F.rank().over(windowSpec))
display(ranked_user)



In [0]:
#Filter top 3 users
top_3 = ranked_user.filter(F.col("rank") <= 3).drop("rank")
display(top_3)